<a href="https://colab.research.google.com/github/anthony-yt/deepwater-image-segmentation-classical-quantum/blob/feat%2Funet-ablation/notebooks/3_3_unet_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 5.11 - Ablacion del U-Net: escala, perdida y atencion

**Objetivo.** Identificar que factor mejora un modelo pequeno con pocos datos y cerrar el Goal 2.3
original (Attention U-Net). La pregunta concreta: **si existe una mejora, supera la variacion
entre semillas?**

**Protocolo.** Cuatro configuraciones, **una sola variable cambiada por vez** respecto a la base,
cada una con **5 semillas**. Mismo split, mismas transformaciones base, mismo numero de epocas.

| id | que cambia respecto a la base |
|---|---|
| `A_base` | nada: U-Net, CE + Dice, sin augmentacion de escala, sin atencion |
| `B_escala` | anade `zoom_in_out_transforms(0.3, 1.5)` de la task 1.2 |
| `C_tversky` | Tversky (alpha 0,7, beta 0,3) en lugar de CE + Dice |
| `D_atencion` | `attention=True` (Attention U-Net) |

**Metricas.** Las del arnes compartido de la task 5.7 (`src/evaluation/segmentation_metrics.py`),
con la linea base "todo estructura" al lado y agregacion entre semillas con `across_seeds`.

---

## Advertencia sobre la interpretacion

Esta ablacion **no es interpretable con las etiquetas actuales**, y hay que decirlo antes de
ejecutarla.

`mmseg_pipes` se construye desde `SAM_Masks_Frames_CLAHE`. La task 4.3
(`2_3_sam_mask_audit.ipynb`) mide esas mascaras contra el ground truth curado y obtiene IoU 0,664
en Video1 contra una linea base de 0,718, y IoU 0,083 en Video6, donde la mascara marca el 95 % del
frame mientras la estructura real ocupa el 8 %. La task 4.2
(`3_2_unet_training_diagnosis.ipynb`) muestra que, sobre esas etiquetas, el U-Net entrenado iguala
exactamente la prediccion constante: IoU 0,7025 frente a 0,7031.

Con ese techo, las cuatro configuraciones mediran la misma constante cuatro veces y la ablacion no
puede distinguir un factor util de otro inutil.

**Como usar este notebook hoy:** ejecutarlo con `FAST = True` y `SEEDS = [0]` como prueba de que el
codigo corre de punta a punta, y dejar la corrida completa para cuando la task 5.1 termine las
mascaras curadas de los 7 videos y `mmseg_pipes` se regenere desde `Test_Set_Curado`. Hoy hay 28
de las 68 mascaras previstas (Video1 con 20 y Video6 con 8).

**Tambien hay que acordar el split con Dayana** antes de la corrida definitiva: el reparto actual
mezcla frames del mismo video entre entrenamiento y validacion (19 de 21 frames de validacion
tienen un vecino temporal inmediato en entrenamiento), asi que debe sustituirse por el
*leave-one-video-out* de la task 5.4.

## 1. Entorno y dependencias entre ramas

In [ ]:
import json
import os
import random
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

REPO_URL = "https://github.com/anthony-yt/deepwater-image-segmentation-classical-quantum.git"
BRANCH = "feat/unet-ablation"

# Esta task depende de modulos que viven en otras ramas sin mergear:
#   src/preprocessing/scale_augmentation.py  ->  feat/scale-augmentation   (task 1.2)
#   src/evaluation/segmentation_metrics.py   ->  feat/labelling-evaluation-tools (task 5.7)
# Se traen por checkout explicito, para que la dependencia quede registrada y no copiada.
DEPENDENCIAS = [
    ("feat/scale-augmentation", "src/preprocessing/scale_augmentation.py"),
    ("feat/labelling-evaluation-tools", "src/evaluation/segmentation_metrics.py"),
]

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Under_Water_Dataset")
    if not Path("/content/repo/.git").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    for rama, archivo in DEPENDENCIAS:
        subprocess.run(["git", "fetch", "-q", "origin", rama], check=True)
        subprocess.run(["git", "checkout", f"origin/{rama}", "--", archivo], check=True)
        print("traido", archivo, "de", rama)
else:
    DRIVE_ROOT = Path(os.environ.get("DW_DATA_ROOT", "Under_Water_Dataset")).resolve()
sys.path.insert(0, str(Path.cwd()))

for _, archivo in DEPENDENCIAS:
    assert Path(archivo).exists(), (
        f"falta {archivo}. En local: git checkout origin/<rama> -- {archivo}")

PROCESSED = DRIVE_ROOT / "Dataset_Processed"
MMSEG_ROOT = PROCESSED / "mmseg_pipes"
OUT_DIR = PROCESSED / "unet_ablation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 512
BATCH = 4
LR = 1e-4
EPOCHS = 40
SEEDS = [0, 1, 2, 3, 4]
FAST = True    # True: 2 epocas y una sola semilla. Ponlo en False para la corrida definitiva.

if FAST:
    EPOCHS, SEEDS = 2, [0]
print(f"epocas={EPOCHS}  semillas={SEEDS}  salida={OUT_DIR}")

## 2. El fallo de `ignore_index` que hay que corregir antes de medir la escala

`zoom_in_out_transforms` rellena con `mask_fill=IGNORE_INDEX=255` los pixeles que aparecen al
hacer zoom-out, para marcarlos como "no evaluar". El dataset original de
`3_1_unet_baseline_training.ipynb` hacia:

```python
mask = (mask > 0).astype(np.uint8)     # antes de transformar
...
ce = nn.CrossEntropyLoss()             # sin ignore_index
```

Con eso, el relleno de valor 255 pasa el filtro `> 0` y **cuenta como estructura**. La
configuracion `B_escala` saldria peor que la base por un error de implementacion y no por la
augmentacion, que es justo la conclusion que la tarea quiere evitar.

`src/data/pipe_dataset.py` binariza a {0, 1} *antes* de transformar, de modo que el 255 que
introduce el padding sobrevive como valor de ignore, y las perdidas de `src/training/losses.py`
declaran `ignore_index=255`. La celda siguiente lo comprueba.

In [ ]:
import albumentations as A
import torch
from torch.utils.data import DataLoader

from src.data.pipe_dataset import IGNORE_INDEX, PipeDataset, list_pairs
from src.evaluation.segmentation_metrics import across_seeds, evaluate
from src.models.unet import build_unet
from src.preprocessing.scale_augmentation import zoom_in_out_transforms
from src.training.losses import make_criterion
from src.training.loop import baseline_metrics, train_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| dispositivo", DEVICE)

BASE_AUG = [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(0.1, 0.1, p=0.3)]

tf_sin_escala = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)] + BASE_AUG)
tf_con_escala = A.Compose(zoom_in_out_transforms(IMG_SIZE, (0.3, 1.5)) + BASE_AUG)
tf_val = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])

train_pairs = list_pairs(MMSEG_ROOT, "training")
val_pairs = list_pairs(MMSEG_ROOT, "validation")
assert train_pairs and val_pairs, f"no hay pares en {MMSEG_ROOT}"
print(f"train {len(train_pairs)} | val {len(val_pairs)}")

In [ ]:
# Comprobacion del ignore: con zoom-out deben aparecer pixeles 255 y nunca otro valor.
ds = PipeDataset(train_pairs, tf_con_escala)
valores = set()
n_con_ignore = 0
for i in range(min(40, len(ds))):
    _, mask = ds[i]
    v = set(torch.unique(mask).tolist())
    valores |= v
    n_con_ignore += int(IGNORE_INDEX in v)

print("valores observados en las mascaras:", sorted(valores))
assert valores <= {0, 1, IGNORE_INDEX}, f"valor inesperado en la mascara: {sorted(valores)}"
print(f"muestras con padding de ignore: {n_con_ignore} de {min(40, len(ds))}")

# La perdida debe ignorarlos: una prediccion perfecta sobre los pixeles validos da perdida ~0
# aunque el relleno de ignore este presente y mal predicho.
_, mask = next((m for m in (ds[i] for i in range(len(ds))) if IGNORE_INDEX in m[1]), (None, None))
if mask is None:
    print("ninguna muestra trajo padding; sube el rango de zoom-out para probarlo")
else:
    valid = mask != IGNORE_INDEX
    logits = torch.zeros(1, 2, *mask.shape)
    logits[0, 1][(mask == 1) & valid] = 20.0     # acierta estructura
    logits[0, 0][(mask == 0) & valid] = 20.0     # acierta fondo
    logits[0, 0][~valid] = 20.0                  # en los ignorados predice fondo a proposito
    for nombre in ["ce", "ce_dice", "focal", "tversky"]:
        valor = float(make_criterion(nombre)(logits, mask.unsqueeze(0)))
        print(f"  {nombre:8s} con prediccion perfecta en los validos: {valor:.5f}")

## 3. Configuraciones

Cada fila cambia exactamente una cosa respecto a `A_base`.

In [ ]:
CONFIGS = {
    "A_base":     dict(transform=tf_sin_escala, loss="ce_dice", attention=False),
    "B_escala":   dict(transform=tf_con_escala, loss="ce_dice", attention=False),
    "C_tversky":  dict(transform=tf_sin_escala, loss="tversky", attention=False),
    "D_atencion": dict(transform=tf_sin_escala, loss="ce_dice", attention=True),
}

print(pd.DataFrame([
    {"config": k, "escala": v["transform"] is tf_con_escala, "perdida": v["loss"],
     "atencion": v["attention"]}
    for k, v in CONFIGS.items()]).set_index("config").to_string())

# Una sola variable distinta respecto a la base, por construccion.
base = CONFIGS["A_base"]
for nombre, cfg in CONFIGS.items():
    if nombre == "A_base":
        continue
    distintas = [k for k in base if cfg[k] != base[k]]
    assert len(distintas) == 1, f"{nombre} cambia {distintas}, deberia cambiar una sola variable"
print("\ncada configuracion cambia exactamente una variable respecto a A_base")

## 4. Prediccion y evaluacion por video

El modelo trabaja a 512x512 y las mascaras de referencia estan a su resolucion original
(640x360, 480x360 o 640x352 segun el video). `predict_masks` de `src/training/predict.py` devuelve
la prediccion al tamano del frame con vecino mas cercano antes de medir, porque medir a 512x512
mezclaria el error del modelo con el del reescalado. Es la decision 7 del protocolo
(`docs/protocolo_evaluacion.md`) y la comparte con la task 4.8.

In [ ]:
from src.training.predict import predict_masks


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print("linea base del split de validacion:")
val_dl_check = DataLoader(PipeDataset(val_pairs, tf_val), batch_size=BATCH, shuffle=False)
print(json.dumps(baseline_metrics(val_dl_check), indent=2))

## 5. Corrida

4 configuraciones x las semillas de `SEEDS`. Con `FAST = False` y 5 semillas son 20
entrenamientos de 40 epocas: planifica la sesion de Colab en consecuencia y vuelve a ejecutar solo
las configuraciones que falten, porque los resultados por semilla se guardan en Drive.

In [ ]:
per_frame_por_config = {}
resumen = []

for nombre, cfg in CONFIGS.items():
    train_dl = DataLoader(PipeDataset(train_pairs, cfg["transform"]), batch_size=BATCH,
                          shuffle=True, num_workers=2)
    val_dl = DataLoader(PipeDataset(val_pairs, tf_val), batch_size=BATCH, shuffle=False,
                        num_workers=2)
    per_frame_por_config[nombre] = {}

    for seed in SEEDS:
        destino = OUT_DIR / f"per_frame_{nombre}_seed{seed}.csv"
        if destino.exists():
            per_frame_por_config[nombre][seed] = pd.read_csv(destino)
            print(f"{nombre} semilla {seed}: reutilizado de {destino.name}")
            continue

        print(f"\n{'=' * 70}\n{nombre} | semilla {seed}\n{'=' * 70}")
        set_seed(seed)
        model = build_unet(attention=cfg["attention"], device=DEVICE)
        result = train_model(model, train_dl, val_dl, make_criterion(cfg["loss"]), DEVICE,
                             epochs=EPOCHS, lr=LR, scheduler="cosine", eval_train=False,
                             select_on="dice", log_every=10)
        model.load_state_dict(result["best_state"])

        preds, gts = predict_masks(model, val_pairs, DEVICE)
        per_frame, per_video = evaluate(preds, gts, n_boot=500)
        per_frame.to_csv(destino, index=False)
        per_frame_por_config[nombre][seed] = per_frame

        resumen.append({"config": nombre, "seed": seed, "mejor_epoca": result["best_epoch"],
                        "iou": per_frame.iou.mean(), "base_iou": per_frame.base_iou.mean(),
                        "boundary_f1": per_frame.boundary_f1.mean(),
                        "minutos": round(result["seconds"] / 60, 1)})
        print(per_video[["video", "n_frames", "iou", "base_iou", "iou_minus_base",
                         "boundary_f1"]].round(4).to_string(index=False))
        del model, result
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

if resumen:
    print("\n--- corridas nuevas de esta sesion ---")
    print(pd.DataFrame(resumen).round(4).to_string(index=False))

## 6. Tabla de resultados

Cuatro filas por video, con media y desviacion entre semillas, y la linea base al lado. El arnes
de la task 5.7 lo produce con `across_seeds`.

In [ ]:
bloques = []
for nombre, por_semilla in per_frame_por_config.items():
    if not por_semilla:
        continue
    tabla = across_seeds(por_semilla).reset_index()
    tabla.insert(0, "config", nombre)
    bloques.append(tabla)

ablacion = pd.concat(bloques, ignore_index=True)
cols = ["config", "video", "n_seeds", "iou", "iou_sd", "base_iou", "boundary_f1",
        "boundary_f1_sd", "dice", "dice_sd"]
cols = [c for c in cols if c in ablacion.columns]
print(ablacion[cols].round(4).to_string(index=False))
ablacion.to_csv(OUT_DIR / "ablacion_5_11.csv", index=False)
print("\nguardado en", OUT_DIR / "ablacion_5_11.csv")

In [ ]:
# La pregunta del objetivo: alguna mejora supera la variacion entre semillas?
pivote = ablacion.pivot_table(index="video", columns="config", values="iou")
sd = ablacion.pivot_table(index="video", columns="config", values="iou_sd")
base = ablacion.pivot_table(index="video", columns="config", values="base_iou").iloc[:, 0]

print("--- IoU medio por video y configuracion ---")
print(pivote.round(4).to_string())
print("\n--- desviacion entre semillas ---")
print(sd.round(4).to_string())

if "A_base" in pivote.columns:
    print("\n--- diferencia respecto a A_base, y si excede la variacion entre semillas ---")
    for config in [c for c in pivote.columns if c != "A_base"]:
        delta = pivote[config] - pivote["A_base"]
        # Criterio: la diferencia debe superar la suma de las desviaciones de las dos corridas.
        umbral = sd[config].fillna(0) + sd["A_base"].fillna(0)
        for video in pivote.index:
            veredicto = ("supera el ruido" if abs(delta[video]) > umbral[video]
                         else "dentro del ruido")
            print(f"  {config:12s} {video:8s} delta={delta[video]:+.4f}  "
                  f"umbral={umbral[video]:.4f}  -> {veredicto}")
    print("\n--- linea base todo estructura por video ---")
    print(base.round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
ancho = 0.2
videos = list(pivote.index)
x = np.arange(len(videos))
for i, config in enumerate(pivote.columns):
    ax.bar(x + i * ancho, pivote[config].values, ancho,
           yerr=sd[config].fillna(0).values, capsize=3, label=config)
ax.plot(x + ancho * (len(pivote.columns) - 1) / 2, base.values, "k_", markersize=26,
        label="todo estructura")
ax.set_xticks(x + ancho * (len(pivote.columns) - 1) / 2)
ax.set_xticklabels(videos)
ax.set_ylabel("IoU")
ax.set_title("Ablacion del U-Net por video. Las barras de error son la desviacion entre semillas")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Conclusion

*Se completa tras la corrida definitiva, que requiere las etiquetas de la task 5.1.*

### Como leer la tabla

Para cada configuracion y cada video hay tres numeros que importan: el IoU medio entre semillas,
la desviacion entre semillas y la linea base "todo estructura" de ese video. Una configuracion
mejora de verdad solo si **las dos** condiciones se cumplen:

1. Su diferencia respecto a `A_base` excede la variacion entre semillas de las dos corridas.
2. Su IoU supera la linea base del video.

La segunda condicion no es formal. En Video1 la linea base es 0,718 y en Video6 es 0,081: un IoU
de 0,70 es un resultado excelente en Video6 y un empate con no hacer nada en Video1. Cualquier
umbral absoluto de mejora, como los 2,0 puntos de IoU que fija el documento de diseno hibrido,
cambia de significado segun el video.

### Lo que ya se puede afirmar

Sobre las etiquetas actuales la ablacion no puede responder a la pregunta del objetivo. La task
4.2 muestra que el U-Net entrenado iguala la prediccion constante con una diferencia de -0,0006 de
IoU, asi que las cuatro configuraciones partiran del mismo techo y las diferencias que aparezcan
seran ruido de semilla. Ejecutar la version completa antes de regenerar las etiquetas gasta unas
20 corridas de GPU para medir cuatro veces la misma constante.

### Precondiciones de la corrida definitiva

1. La task 5.1 completa: las 68 mascaras curadas de los 7 videos en `Test_Set_Curado`. Hoy hay 28.
2. `mmseg_pipes` regenerado desde `Test_Set_Curado` y no desde `SAM_Masks_Frames_CLAHE`.
3. Split por video acordado con Dayana, segun el protocolo *leave-one-video-out* de la task 5.4,
   con las mismas 5 semillas en las dos tasks para que la tabla de la task 4.8 sea comparable.
4. `FAST = False` y `SEEDS = [0, 1, 2, 3, 4]`.